# Module 1.4 — Functions & Scope

### Python Mastery · Track 1: Python Fundamentals

Functions let you name a piece of work, reuse it, and reason about your program in manageable pieces. This module covers how to define functions, the several ways to pass arguments, how Python decides which variables are visible where, and the compact `lambda` form.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Change the arguments and definitions and run again.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Define functions with parameters, default values, and return values.
2. Distinguish positional from keyword arguments and use `*args` and `**kwargs`.
3. Use positional-only (`/`) and keyword-only (`*`) parameters.
4. Explain local versus global scope and use `global` and `nonlocal` correctly.
5. Write small anonymous functions with `lambda`.

**Prerequisites:** Modules 1.1 to 1.3.

---

## Part 1 · Defining Functions

A function is defined with `def`, a name, a parenthesised list of parameters, and an indented body. The `return` statement sends a value back to the caller. A function that does not execute a `return` returns `None` automatically.

Parameters may have **default values**, which are used when the caller omits that argument.

In [ ]:
def greet(name, greeting="Hello"):
    """Return a greeting for the given name."""
    return greeting + ", " + name + "."

print(greet("Ada"))                 # uses the default greeting
print(greet("Ada", "Welcome"))      # overrides the default

def nothing():
    pass

print("A function with no return gives:", nothing())   # None

A function can return several values at once by separating them with commas; Python packs them into a tuple, which the caller can unpack.

In [ ]:
def min_and_max(numbers):
    """Return the smallest and largest of a list of numbers."""
    return min(numbers), max(numbers)

low, high = min_and_max([4, 9, 1, 7])    # unpack the returned pair
print("low:", low, "high:", high)

## Part 2 · Positional, Keyword, `*args`, and `**kwargs`

Arguments can be passed **by position** (matched in order) or **by keyword** (matched by name). Keyword arguments improve readability and let you skip earlier defaults.

`*args` collects any extra positional arguments into a tuple. `**kwargs` collects any extra keyword arguments into a dictionary. Together they let a function accept a flexible number of inputs.

In [ ]:
def describe(label, *values, **options):
    print("label:", label)
    print("values (tuple):", values)
    print("options (dict):", options)

# 'label' is positional; the next three become 'values'; the named ones become 'options'.
describe("scores", 90, 85, 77, sort=True, reverse=False)

In [ ]:
def total(*numbers):
    """Sum any number of arguments."""
    result = 0
    for n in numbers:
        result += n
    return result

print(total(1, 2, 3))
print(total(10, 20, 30, 40))

# The * can also unpack a list into positional arguments at the call site.
data = [5, 5, 5]
print(total(*data))

## Part 3 · Positional-only and Keyword-only Parameters (Python 3.8+)

You can control **how** callers are allowed to pass arguments:

- Parameters before a bare `/` are **positional-only**: they cannot be passed by name.
- Parameters after a bare `*` are **keyword-only**: they must be passed by name.

This makes interfaces clearer and lets you rename internal parameters safely.

In [ ]:
def make_box(width, height, /, *, color="white", filled=False):
    # width and height are positional-only; color and filled are keyword-only.
    return f"{width}x{height} box, color={color}, filled={filled}"

print(make_box(3, 4))                          # positional width/height
print(make_box(3, 4, color="red", filled=True))

# make_box(width=3, height=4) would fail: width/height cannot be keywords.
# make_box(3, 4, "red") would fail: color must be passed by name.

## Part 4 · Scope: Local, Global, and `nonlocal`

A variable's **scope** is the region where it is visible. Python resolves names using the LEGB rule: Local, then Enclosing function, then Global (module), then Built-in.

By default, assigning to a name inside a function creates a **local** variable. To rebind a module-level (global) name from inside a function, declare it `global`. To rebind a name from an enclosing function, declare it `nonlocal`.

In [ ]:
counter = 0          # a global variable

def increment():
    global counter   # without this line, the assignment would create a new local name
    counter += 1

increment()
increment()
print("counter is now", counter)

In [ ]:
def make_accumulator():
    """Return a function that remembers a running total using nonlocal."""
    running = 0
    def add(amount):
        nonlocal running        # rebind the enclosing 'running', not a new local one
        running += amount
        return running
    return add

acc = make_accumulator()
print(acc(10))   # 10
print(acc(5))    # 15
print(acc(100))  # 115

> **Caution — the mutable default argument trap.** A default value is created **once**, when the function is defined, not on each call. A mutable default such as `[]` is therefore shared across calls, which is almost never intended. Use `None` as the default and create a fresh object inside the function.

In [ ]:
# Buggy version: the same list is reused across calls.
def add_item_buggy(item, bag=[]):
    bag.append(item)
    return bag

print(add_item_buggy("a"))   # ['a']
print(add_item_buggy("b"))   # ['a', 'b']  <- surprise: the list persisted

# Correct version:
def add_item(item, bag=None):
    if bag is None:
        bag = []
    bag.append(item)
    return bag

print(add_item("a"))         # ['a']
print(add_item("b"))         # ['b']  <- a fresh list each time

## Part 5 · `lambda` — Small Anonymous Functions

A `lambda` is a one-expression function written inline, without a name. It is handy where a small function is needed briefly, most commonly as a `key` for sorting. For anything longer than a single expression, prefer a normal `def`.

In [ ]:
square = lambda x: x * x        # equivalent to a one-line def
print(square(6))

people = [("Ada", 36), ("Ben", 28), ("Cara", 41)]
people.sort(key=lambda person: person[1])   # sort by the second item (age)
print(people)

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A function with a default argument (Easy)

In [ ]:
def power(base, exponent=2):
    """Raise base to exponent; squares by default."""
    return base ** exponent

print(power(5))        # 25, using the default exponent
print(power(2, 10))    # 1024
# Experiment: call power with a different exponent.

### Example 2 — Returning multiple values (Easy)

In [ ]:
def divide(a, b):
    """Return the quotient and remainder of a divided by b."""
    return a // b, a % b

q, r = divide(17, 5)
print("quotient:", q, "remainder:", r)
# Experiment: change the inputs.

### Example 3 — Collecting arguments with `*args` and `**kwargs` (Medium)

In [ ]:
def build_profile(name, *hobbies, **details):
    print("Name:", name)
    print("Hobbies:", hobbies)
    for key, value in details.items():
        print(f"  {key}: {value}")

build_profile("Maya", "reading", "cycling", city="Pune", age=29)
# Experiment: add or remove hobbies and details.

### Example 4 — Keyword-only and positional-only parameters (Medium)

In [ ]:
def connect(host, port, /, *, timeout=30, secure=True):
    return f"connecting to {host}:{port} (timeout={timeout}, secure={secure})"

print(connect("example.com", 443))
print(connect("example.com", 80, timeout=10, secure=False))
# host and port must be positional; timeout and secure must be named.

### Example 5 — A closure that remembers state, plus the default-argument fix (Difficult)

In [ ]:
def make_counter(start=0):
    """Return a function that increments and returns a private counter."""
    count = start
    def step(by=1):
        nonlocal count
        count += by
        return count
    return step

c = make_counter(10)
print(c())     # 11
print(c(5))    # 16

# A safe accumulator using the None-default pattern:
def collect(item, into=None):
    if into is None:
        into = []
    into.append(item)
    return into

print(collect("x"))    # ['x']
print(collect("y"))    # ['y'] each call starts fresh

### Example 6 — Higher-order functions and `lambda` (Difficult)

In [ ]:
def apply_to_each(values, transform):
    """Return a new list with transform applied to every value."""
    return [transform(v) for v in values]

nums = [1, 2, 3, 4]
print(apply_to_each(nums, lambda x: x * x))     # squares
print(apply_to_each(nums, lambda x: x + 100))   # add 100

# Sorting strings by length using a lambda key:
words = ["banana", "fig", "cherry", "kiwi"]
print(sorted(words, key=lambda w: len(w)))

---

## Exercises

**Exercise 1 (Easy).** Write a function `average(*numbers)` that returns the mean of any quantity of numbers. Test it with two different calls.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write a function `repeat(text, times=2)` that returns `text` repeated `times` times with a space between copies. Call it with and without the second argument.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a function `log(message, *, level="INFO")` where `level` is keyword-only. It should return a string such as `[INFO] message`. Demonstrate calling it with a different level.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** The function below has the mutable-default-argument bug. Rewrite it so each call starts with an empty list unless one is supplied.

In [ ]:
def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags
# Rewrite below:


**Exercise 5 (Difficult).** Given the list of dictionaries below, use `sorted` with a `lambda` key to sort the people by age, youngest first, and print the result.

In [ ]:
people = [{"name": "A", "age": 40}, {"name": "B", "age": 25}, {"name": "C", "age": 33}]
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
def average(*numbers):
    if not numbers:
        return 0
    return sum(numbers) / len(numbers)

print(average(2, 4, 6))        # 4.0
print(average(10, 20, 30, 40)) # 25.0

**Solution 2**

In [ ]:
def repeat(text, times=2):
    return " ".join([text] * times)

print(repeat("hi"))            # hi hi
print(repeat("go", 3))         # go go go

**Solution 3**

In [ ]:
def log(message, *, level="INFO"):
    return f"[{level}] {message}"

print(log("server started"))
print(log("disk almost full", level="WARNING"))

**Solution 4**

In [ ]:
def add_tag(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags

print(add_tag("python"))   # ['python']
print(add_tag("data"))     # ['data'] a fresh list each time

**Solution 5**

In [ ]:
people = [{"name": "A", "age": 40}, {"name": "B", "age": 25}, {"name": "C", "age": 33}]
ordered = sorted(people, key=lambda person: person["age"])
print(ordered)

---

## Summary and Key Points

- Functions are defined with `def`; `return` sends back a value, and the absence of `return` yields `None`. Multiple values are returned as a tuple.
- Arguments pass by position or keyword. `*args` gathers extra positional arguments into a tuple; `**kwargs` gathers extra keyword arguments into a dict. A `*` at a call site unpacks a sequence into arguments.
- A bare `/` marks preceding parameters as positional-only; a bare `*` marks following parameters as keyword-only.
- Names resolve by the LEGB rule. Assignment inside a function is local unless declared `global` (module level) or `nonlocal` (enclosing function). Avoid mutable default arguments; use `None` and create the object inside.
- `lambda` defines a small anonymous one-expression function, most useful as a sort `key`.

### Next module: 1.5 — Error Handling

The next module covers responding to things that go wrong: `try`/`except`/`else`/`finally`, raising and chaining exceptions, custom exception classes, and exception groups.